# Feature Extraction or Text Representation 

## Types:
1. Frequency based ex.. BOW , N-grams etc..
2. Prediction based using DL algo's eg.. Word2Vec

## Word Frequency Distribution

In [2]:
from collections import Counter
import nltk
from nltk import word_tokenize, sent_tokenize
# nltk.download("punkt_tab")

In [3]:
def word_frequency(text):
    tokens = word_tokenize(text.lower())
    print(tokens)
    return Counter(tokens)

In [4]:
text = text = "NLP is amazing. NLP makes machines understand text."
word_frequency(text)

['nlp', 'is', 'amazing', '.', 'nlp', 'makes', 'machines', 'understand', 'text', '.']


Counter({'nlp': 2,
         '.': 2,
         'is': 1,
         'amazing': 1,
         'makes': 1,
         'machines': 1,
         'understand': 1,
         'text': 1})

In [5]:
# Here punctuations are includede n
# but it will be removed in text preprocessing step after that we do text representation
# we can use spacy to remove punctuations as well
# generally we remove punctuations after tokenization

In [6]:
import spacy 
nlp = spacy.load("en_core_web_sm")

In [7]:
def word_frequency(text):
    doc = nlp(text.lower())
    tokens = []
    for token in doc:
    
        if not token.is_punct:
            tokens.append(token.text)
    print(tokens)
    return Counter(tokens)

In [8]:
text = text = "NLP is amazing. NLP makes machines understand text. right?"
word_frequency(text)

['nlp', 'is', 'amazing', 'nlp', 'makes', 'machines', 'understand', 'text', 'right']


Counter({'nlp': 2,
         'is': 1,
         'amazing': 1,
         'makes': 1,
         'machines': 1,
         'understand': 1,
         'text': 1,
         'right': 1})

**Now it is correct**

## 1. OHE : OneHotEncoding

In [9]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [10]:
corpus = [
    "NLP is fun and exciting",
    "Machines understand NLP and text",
    "Text processing is part of NLP"
]

In [11]:
df = pd.DataFrame(corpus, columns=["text"])

In [12]:
df.head()

,text
0,NLP is fun and exciting
1,Machines understand NLP and text
2,Text processing is part of NLP


In [13]:
def word_tokenize_spacy(text):
    doc = nlp(text.lower())
    tokens = []
    for token in doc:
        if not token.is_punct:
            tokens.append(token.text)
    return tokens

In [14]:
df['tokens'] = df['text'].apply(word_tokenize_spacy)

In [15]:
df.head()

,text,tokens
0,NLP is fun and exciting,"[nlp, is, fun, and, exciting]"
1,Machines understand NLP and text,"[machines, understand, nlp, and, text]"
2,Text processing is part of NLP,"[text, processing, is, part, of, nlp]"


In [16]:
df['word_frequency'] = df['text'].apply(lambda x:Counter(word_tokenize_spacy(x)))

In [17]:
df.head()

,text,tokens,word_frequency
0,NLP is fun and exciting,"[nlp, is, fun, and, exciting]","{'nlp': 1, 'is': 1, 'fun': 1, 'and': 1, 'excit..."
1,Machines understand NLP and text,"[machines, understand, nlp, and, text]","{'machines': 1, 'understand': 1, 'nlp': 1, 'an..."
2,Text processing is part of NLP,"[text, processing, is, part, of, nlp]","{'text': 1, 'processing': 1, 'is': 1, 'part': ..."


In [18]:
def build_vocab(series):
    vocab = []
    for sentence in series:
        doc = nlp(sentence.lower())
        for token in doc:
            if not token.is_punct and token.text not in vocab:
                vocab.append(token.text)
    return vocab

In [19]:
vocab = build_vocab(df.text)
print(vocab)

['nlp', 'is', 'fun', 'and', 'exciting', 'machines', 'understand', 'text', 'processing', 'part', 'of']


In [20]:
def OHE(text):
    document = []
    vector = np.zeros(len(vocab))
    for word in word_tokenize(text.lower()):
        if word not in vocab:
            document.append(vector)
        else:
            index = vocab.index(word)
            vector[index] = 1
            document.append(vector)
            vector = np.zeros(len(vocab))
    return document

In [21]:
df["OHE"] = df.text.apply(OHE)

In [22]:
df.text[0]

'NLP is fun and exciting'

In [23]:
df.OHE[0]

[array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.]),
 array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]),
 array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])]

### Pros
- intuitive logic
- easy implementation

### Cons
- Sparsity , if i have 50k vocab then every word will be having 49999 zeros and 1 one. and it lead to overfitting in ML
- shape of every document is not fixed but ML algo assume fixed input
- in prediction if new word comes which is not there even in vocab , at that time it will ignore that word, but that word might have more weightage in that sentence.(OOB - Out Of Vocab)
- it doesn't capture semantic meaning at all

## 2. BOW - Bag of words
- based on frequency of words, how many times a word came in that document (remember sentence --> document)
- mostly used in text classification problems

In [24]:
corpus

['NLP is fun and exciting',
 'Machines understand NLP and text',
 'Text processing is part of NLP']

In [25]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [26]:
bow = cv.fit_transform(corpus)

In [27]:
print("Vocabulary Built:", cv.vocabulary_)
print("Feature names:", cv.get_feature_names_out())

Vocabulary Built: {'nlp': 5, 'is': 3, 'fun': 2, 'and': 0, 'exciting': 1, 'machines': 4, 'understand': 10, 'text': 9, 'processing': 8, 'part': 7, 'of': 6}
Feature names: ['and' 'exciting' 'fun' 'is' 'machines' 'nlp' 'of' 'part' 'processing'
 'text' 'understand']


In [28]:
# BOW --> feature extracted or text representation
bow.toarray()

array([[1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1],
       [0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0]])

In [29]:
X = (cv.fit_transform(df['text'])).toarray()

In [30]:
X

array([[1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1],
       [0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0]])

In [31]:
df_bow = pd.DataFrame(X, columns = cv.get_feature_names_out())

In [32]:
df_bow.head()

,and,exciting,fun,is,machines,nlp,of,part,processing,text,understand
0,1,1,1,1,0,1,0,0,0,0,0
1,1,0,0,0,1,1,0,0,0,1,1
2,0,0,0,1,0,1,1,1,1,1,0


### 2.1 Binary BOW

In [33]:
cv1 = CountVectorizer(binary=True)
bow = cv.fit_transform(corpus)
bow.toarray()

array([[1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1],
       [0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0]])

- binary = True --> all non zero elements become 1 , better performing in sentiment analysis
- called binary BOW , it does count words it simpley if it present then 1 else 0

### Advantages
- intuitive
- solves fixed size problem of OHE
- OOV problem solved
- it captures semantic meaning at some extent 

### Disadvantages
- sparsity
- if Hello word comes in prediction which is not there in vocab so it says it zero but this word Hello may be the stronges word among all the words in a document so it is ignoring this word.
- we don't consider order of words
- according to BOW , You are good and You are bad. both are neighbour but in reality both are opposite.

## 3. N-grams
**Instead of forming vocabulary using one word only we consider N-words and form vocab.**
- Bi-grams - 2 words vocab
- tri-grams - 3 words vocab
- N-grams  - N words vocab

In [34]:
df.head()

,text,tokens,word_frequency,OHE
0,NLP is fun and exciting,"[nlp, is, fun, and, exciting]","{'nlp': 1, 'is': 1, 'fun': 1, 'and': 1, 'excit...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1,Machines understand NLP and text,"[machines, understand, nlp, and, text]","{'machines': 1, 'understand': 1, 'nlp': 1, 'an...","[[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0,..."
2,Text processing is part of NLP,"[text, processing, is, part, of, nlp]","{'text': 1, 'processing': 1, 'is': 1, 'part': ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0,..."


In [35]:
cv_bigram = CountVectorizer(ngram_range=(2,2))
cv_bigram.fit_transform(corpus).toarray()

array([[1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0]])

In [36]:
cv_bigram.vocabulary_

{'nlp is': 7,
 'is fun': 3,
 'fun and': 2,
 'and exciting': 0,
 'machines understand': 5,
 'understand nlp': 12,
 'nlp and': 6,
 'and text': 1,
 'text processing': 11,
 'processing is': 10,
 'is part': 4,
 'part of': 9,
 'of nlp': 8}

In [37]:
cv_bigram.get_feature_names_out()

array(['and exciting', 'and text', 'fun and', 'is fun', 'is part',
       'machines understand', 'nlp and', 'nlp is', 'of nlp', 'part of',
       'processing is', 'text processing', 'understand nlp'], dtype=object)

In [38]:
cv_unigram_bigram = CountVectorizer(ngram_range=(1, 2))
cv_unigram_bigram.fit_transform(corpus).toarray()

array([[1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0],
       [1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        1, 1],
       [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        0, 0]])

In [39]:
cv_unigram_bigram.get_feature_names_out()

array(['and', 'and exciting', 'and text', 'exciting', 'fun', 'fun and',
       'is', 'is fun', 'is part', 'machines', 'machines understand',
       'nlp', 'nlp and', 'nlp is', 'of', 'of nlp', 'part', 'part of',
       'processing', 'processing is', 'text', 'text processing',
       'understand', 'understand nlp'], dtype=object)

* **ngram_range**
- (2, 2) : only bigram
- (1, 2) : unigrams and bigrams both
- (1, 3) : unigrams, bigrams and trigrams

### Advantages
- now You are good. and You are not good. both vectors are far away
- intuitive explanation and easy to implement

### Disadvantages
- if dataset is very large then it becomes slow
- OOV is still there , it won't throw error but it ignores new word


## 4. Tf-Idf

- instead of giving same weights to all the words which are present in a document it gives different weights based of TF and IDF calculation
- TF - Term frequency , how is the occurrence of given term in a document 
- TF(t, d) - (no of occurrences of term t in document d)/(total number of terms in document d)
- IDF - inverse document frequency, how is the occurrence of given term(word) in corpus
- IDF(t, d) = loge((total documents in corpus)/(no. of documents with term t in them))
- weight(t, d) = TF(t, d) * IDF(t, d)

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [41]:
tfidf.fit_transform(corpus).toarray()

array([[0.40619178, 0.53409337, 0.53409337, 0.40619178, 0.        ,
        0.31544415, 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.40619178, 0.        , 0.        , 0.        , 0.53409337,
        0.31544415, 0.        , 0.        , 0.        , 0.40619178,
        0.53409337],
       [0.        , 0.        , 0.        , 0.35829137, 0.        ,
        0.27824521, 0.4711101 , 0.4711101 , 0.4711101 , 0.35829137,
        0.        ]])

In [42]:
tfidf.get_feature_names_out()

array(['and', 'exciting', 'fun', 'is', 'machines', 'nlp', 'of', 'part',
       'processing', 'text', 'understand'], dtype=object)

### Advantages
- mostly used in information retrieval systems
- in google
### Disadvantes
- sparsity
- OOV
- dimensionality more --> overfitting
- no semantic meaning capturing

## 5. Custom features
- based on domain , sometimes we create our own features
- example : in sentiment analysis of a product based on reviews
- here we can make features like 1. no of positive words 2. no of negative words etc

- Here i used stopwords, but actually we dont' consider stop words before vectorization.
- u can see this blog as well for text preprocessing and can see github
- https://vinod-codes-ai.blogspot.com/2026/05/python-strings-regex-nlp-core-skills.html
- See my text preprocessing.ipynb notebook :-  https://github.com/vinod-kaumar/NLP-by-vinod

## 6. NLP Task - Keyword Extraction

In [ ]:
import numpy as np
feature_array = np.array(tfidf.get_feature_names_out())
print(feature_array)

importance = np.argsort(X).flatten()[::-1]
print(importance)

keywords = feature_array[importance[:5]]
print("Top Keywords:", keywords)

['and' 'exciting' 'fun' 'is' 'machines' 'nlp' 'of' 'part' 'processing'
 'text' 'understand']
[ 9  8  7  5  6  3 10  4  2  1  0 10  9  4  0  5  8  6  7  3  2  1  2  1
  5  0  3 10  9  8  4  7  6]
Top Keywords: ['text' 'processing' 'part' 'nlp' 'of']


### Assignment
- use movies.csv file which u have created in Data_Acquistion.ipynb of Day-1
- do all the step which are explained above|